In [1]:
from unsloth import FastLanguageModel
import torch

print("Loading LFM2.5-1.2B-Instruct from LiquidAI...")
model, tokenizer = FastLanguageModel.from_pretrained(
    # model_name="unsloth/LFM2.5-1.2B-Instruct",
    model_name="LiquidAI/LFM2.5-1.2B-Instruct",
    max_seq_length=16384, # Choose any for long context!
    load_in_4bit=True,  # 4 bit quantization to reduce memory
    load_in_8bit=False,  # [NEW!] A bit more accurate, uses 2x memory
    load_in_16bit=False,  # [NEW!] Enables 16bit LoRA
    full_finetuning=False,  # [NEW!] We have full finetuning now!
    # token = "hf_...", # use one if using gated models
)
print("Model loaded successfully!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


D:\Cache\conda\envs\finetuner\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0317 09:06:18.762000 11812 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


🦥 Unsloth Zoo will now patch everything to make training faster!
Loading LFM2.5-1.2B-Instruct from LiquidAI...
==((====))==  Unsloth 2026.3.4: Fast Lfm2 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA GeForce RTX 3050 6GB Laptop GPU. Num GPUs = 1. Max memory: 6.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.10.0+cu130. CUDA: 8.6. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|█████| 148/148 [00:05<00:00, 28.36it/s, Materializing param=model.layers.15.operator_norm.weight]


Model loaded successfully!


In [2]:
model = FastLanguageModel.get_peft_model(
    model,
    r=64, # Moderate rank to save VRAM at high context
    target_modules=["q_proj", "k_proj", "v_proj", "out_proj", "in_proj", "w1", "w2", "w3"],
    lora_alpha=128,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

Unsloth: Making `model.base_model.model.model` require gradients


In [3]:
import ast
from datasets import load_dataset, concatenate_datasets
from unsloth.chat_templates import get_chat_template, standardize_data_formats

# dataset = load_dataset("ise-uiuc/Magicoder-Evol-Instruct-110K", split = "train[:15000]")

# Apply ChatML template to tokenizer
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml",
)

Unsloth: Will map <|im_end|> to EOS = <|im_end|>.


In [4]:
from datasets import load_dataset, concatenate_datasets

print("Loading and formatting Magicoder...")
magicoder_ds = load_dataset("ise-uiuc/Magicoder-Evol-Instruct-110K", split="train").select(range(1500))

def format_magicoder(example):
    instr = example.get("instruction", "")
    inp = example.get("input", "")
    content = f"{instr}\n{inp}".strip() if inp else instr
    return {"conversations": [{"role": "user", "content": content}, {"role": "assistant", "content": example.get("response", "")}]}

magicoder_ds = magicoder_ds.map(format_magicoder, remove_columns=magicoder_ds.column_names)

print("Loading and formatting CodeFeedback...")
# CodeFeedback uses 'query' and 'answer' instead of 'instruction' and 'response'
feedback_ds = load_dataset("m-a-p/CodeFeedback-Filtered-Instruction", split="train").select(range(4000))

def format_feedback(example):
    return {"conversations": [{"role": "user", "content": example["query"]}, {"role": "assistant", "content": example["answer"]}]}

feedback_ds = feedback_ds.map(format_feedback, remove_columns=feedback_ds.column_names)

print("Merging into Agentic Mix...")
mixed_dataset = concatenate_datasets([magicoder_ds, feedback_ds]).shuffle(seed=3407)

# Apply ChatML Template
def apply_chatml(example):
    text = tokenizer.apply_chat_template(example["conversations"], tokenize=False, add_generation_prompt=False)
    # Strip BOS token to prevent double-tokenization issues
    return {"text": [t.replace(tokenizer.bos_token, "") if tokenizer.bos_token else t for t in text]}

mixed_dataset = mixed_dataset.map(apply_chatml, batched=True)
print(f"Total rows for training: {len(mixed_dataset)}")

Loading and formatting Magicoder...


Map: 100%|███████████████████████████████████████████████████████████████| 1500/1500 [00:00<00:00, 14408.79 examples/s]


Loading and formatting CodeFeedback...


Map: 100%|███████████████████████████████████████████████████████████████| 4000/4000 [00:00<00:00, 13538.68 examples/s]


Merging into Agentic Mix...


Map: 100%|███████████████████████████████████████████████████████████████| 5500/5500 [00:00<00:00, 16908.23 examples/s]

Total rows for training: 5500


In [5]:
mixed_dataset[5]

{'conversations': [{'content': 'Write a single-line lambda expression that calculates the factorial of a given number, but without using any loops, recursion, or built-in factorial functions. The lambda expression should also be limited to a maximum of 10 characters in length.',
   'role': 'user'},
  {'content': "lambda n: 1 if n<=1 else n * __import__('functools').reduce(lambda x, y: x*y, range(1, n))",
   'role': 'assistant'}],
 'text': "<|im_start|>user\nWrite a single-line lambda expression that calculates the factorial of a given number, but without using any loops, recursion, or built-in factorial functions. The lambda expression should also be limited to a maximum of 10 characters in length.<|im_end|>\n<|im_start|>assistant\nlambda n: 1 if n<=1 else n * __import__('functools').reduce(lambda x, y: x*y, range(1, n))<|im_end|>\n"}

In [4]:
def formatting_prompts_func(examples):
    # Safe extraction of keys to avoid KeyError
    instructions = examples.get("instruction", [])
    inputs       = examples.get("input", [""] * len(instructions)) # Default to empty string if missing
    outputs      = examples.get("response", [])

    texts = []
    for instr, inp, out in zip(instructions, inputs, outputs):
        # Combine instruction and input if input exists
        full_user_msg = f"{instr}\n{inp}".strip() if inp else instr

        convo = [
            {"role": "user", "content": full_user_msg},
            {"role": "assistant", "content": out},
        ]

        # Apply template and clean up
        formatted_text = tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        # Remove the BOS token if it's prepended (common in LFM/Qwen models)
        if tokenizer.bos_token and formatted_text.startswith(tokenizer.bos_token):
            formatted_text = formatted_text.replace(tokenizer.bos_token, "")

        texts.append(formatted_text)

    return { "text" : texts }

# Map the dataset
dataset = dataset.map(formatting_prompts_func, batched = True)

In [5]:
dataset = dataset.map(formatting_prompts_func, batched = True)

In [6]:
dataset[37]["text"]

"<|im_start|>user\nI am writing a program which has two panes (via `CSplitter`), however I am having problems figuring out out to resize the controls in each frame. For simplicity, can someone tell me how I would do it for a basic frame with a single `CEdit` control? \n\nI'm fairly sure it is to do with the `CEdit::OnSize()` function... But I'm not really getting anywhere...\n\nThanks! :)<|im_end|>\n<|im_start|>assistant\nAssuming you're using the MFC `CSplitterWnd` class, here's a basic idea of what you want to do:\n\n(Sidenote: you are correct that you'll need to handle `OnSize()`, but not for `CEdit` -- it has to be handled for the frame in which your `CEdit` control is located.)\n\nLet's move on to code. Assume your `CEdit` control is put in a `CFrameWnd` or `CFormView` subclass, say, `CEditFrame`. In this subclass, you'll have member variable for 'CEdit' control. Let's call it `m_EditControl`.\n\nFirstly, Initialize the `CEdit` control, and set it up how you want it in your frame 

In [8]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = mixed_dataset,
    dataset_text_field = "text",
    max_seq_length = 16384, # Dropped from 16k to 8k to further speed up training!
    args = SFTConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        
        warmup_steps = 10,
        num_train_epochs = 1,
        learning_rate = 2e-4, # Bumped slightly for the new data mix
        
        # --- THE FIXES FOR YOUR ISSUES ---
        save_strategy = "steps", # Enables checkpoint saving
        save_steps = 100,        # Saves every 100 steps!
        lr_scheduler_type = "cosine", # Much better loss curve
        # ---------------------------------
        
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10, # Log more frequently to watch the loss drop
        optim = "adamw_8bit",
        weight_decay = 0.01,
        seed = 3407,
        output_dir = "outputs_agentic_coder",
        report_to = "none",
    ),
)


Unsloth: Tokenizing ["text"]: 100%|████████████████████████████████████████| 5500/5500 [00:08<00:00, 656.46 examples/s]


In [9]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

Filter: 100%|█████████████████████████████████████████████████████████████| 5500/5500 [00:04<00:00, 1226.42 examples/s]


In [10]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA GeForce RTX 3050 6GB Laptop GPU. Max memory = 6.0 GB.
2.174 GB of memory reserved.


In [11]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,500 | Num Epochs = 1 | Total steps = 688
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 44,433,408 of 1,214,774,016 (3.66% trained)


Step,Training Loss
10,0.563178
20,0.517014
30,0.552563
40,0.547023
50,0.536107
60,0.524186
70,0.540318
80,0.496531
90,0.530616
100,0.569080


In [12]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

2628.6273 seconds used for training.
43.81 minutes used for training.
Peak reserved memory = 6.062 GB.
Peak reserved memory for training = 3.888 GB.
Peak reserved memory % of max memory = 101.033 %.
Peak reserved memory for training % of max memory = 64.8 %.


In [13]:
trainer_stats

TrainOutput(global_step=688, training_loss=0.5334520069665687, metrics={'train_runtime': 2628.6273, 'train_samples_per_second': 2.092, 'train_steps_per_second': 0.262, 'total_flos': 2.120207037161472e+16, 'train_loss': 0.5334520069665687, 'epoch': 1.0})

In [14]:
from transformers import TextStreamer

# 1. Prepare the model for faster inference
FastLanguageModel.for_inference(model)

# 2. Define your coding prompt
messages = [
    {"role": "user", 
     "content": "Write a Python script to find the shortest path in a graph using Dijkstra's algorithm without importing any external module.\
     Implement required data structures from scratch."},
]

# 3. Setup the real-time streamer
text_streamer = TextStreamer(tokenizer)

# 4. Tokenize with attention_mask
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
).to("cuda")

# 5. Generate with streaming enabled
print("Generating Response...\n" + "="*20)
_ = model.generate(
    **inputs,
    streamer = text_streamer,    # This shows the code as it's being typed
    max_new_tokens = 512,
    use_cache = True,
    pad_token_id = tokenizer.pad_token_id,

    # Coding-specific optimizations
    temperature = 0.2,           # Lower temperature = more precise code
    repetition_penalty = 1.1,    # Prevents the model from repeating same lines
)

Generating Response...
<|im_start|>user
Write a Python script to find the shortest path in a graph using Dijkstra's algorithm without importing any external module.     Implement required data structures from scratch.<|im_end|>
<|im_start|>assistant
Here is a Python script that implements Dijkstra's algorithm for finding the shortest path in a graph:

```python
import sys

# Function to swap two elements
def swap(arr, i, j):
    arr[i], arr[j] = arr[j], arr[i]

# Function to check if an element exists in an array
def contains(arr, x):
    return x in arr

# Function to get the minimum value from a list of values
def min_value(lst):
    return min(lst)

# Function to add an element to a list
def add_element(lst, x):
    lst.append(x)

# Function to remove an element from a list
def remove_element(lst, x):
    if x in lst:
        lst.remove(x)

# Function to print the graph
def print_graph(graph):
    for node in graph.keys():
        print(node, "->", end="")
        for neighbor in gr

In [15]:
ROOT_PATH = "D:/ML/Models/Finetuned/LiquidAI-LFM2.5-1.2B-Instruct/LiquidAI-LFM2.5-1.2B-Instruct-Code-Pro"
import os
os.makedirs(ROOT_PATH, exist_ok=True)

In [16]:
model.save_pretrained(f"{ROOT_PATH}/lora_r16_4bit_model")  # Local saving
tokenizer.save_pretrained(f"{ROOT_PATH}/lora_r16_4bit_model")

('D:/ML/Models/Finetuned/LiquidAI-LFM2.5-1.2B-Instruct/LiquidAI-LFM2.5-1.2B-Instruct-Code-Pro/lora_r16_4bit_model\\tokenizer_config.json',
 'D:/ML/Models/Finetuned/LiquidAI-LFM2.5-1.2B-Instruct/LiquidAI-LFM2.5-1.2B-Instruct-Code-Pro/lora_r16_4bit_model\\chat_template.jinja',
 'D:/ML/Models/Finetuned/LiquidAI-LFM2.5-1.2B-Instruct/LiquidAI-LFM2.5-1.2B-Instruct-Code-Pro/lora_r16_4bit_model\\tokenizer.json')

In [17]:
model.save_pretrained_gguf(
    f"{ROOT_PATH}/sft_code",
    tokenizer,
    quantization_method = ["q4_k_m", "q8_0", "f16"], 
)

Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: D:\Cache\hf\hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `D:/ML/Models/Finetuned/LiquidAI-LFM2.5-1.2B-Instruct/LiquidAI-LFM2.5-1.2B-Instr


Successfully copied all 1 files from cache to `D:/ML/Models/Finetuned/LiquidAI-LFM2.5-1.2B-Instruct/LiquidAI-LFM2.5-1.2B-Instruct-Code-Pro/sft_code`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|███████████████████████████████████████████████| 1/1 [00:05<00:00,  5.01s/it]


Unsloth: Merge process complete. Saved to `D:\ML\Models\Finetuned\LiquidAI-LFM2.5-1.2B-Instruct\LiquidAI-LFM2.5-1.2B-Instruct-Code-Pro\sft_code`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m', 'q8_0', 'f16'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...


[unsloth_zoo.llama_cpp|WARNING]Unsloth: Qwen2MoE num_experts patch target not found.


Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['D:/ML/Models/Finetuned/LiquidAI-LFM2.5-1.2B-Instruct/LiquidAI-LFM2.5-1.2B-Instruct-Code-Pro/sft_code_gguf\\LFM2.5-1.2B-Instruct.BF16.gguf']
Unsloth: [2] Converting GGUF bf16 into q4_k_m. This might take 10 minutes...
Unsloth: [2] Converting GGUF bf16 into q8_0. This might take 10 minutes...
Unsloth: [2] Converting GGUF bf16 into f16. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['D:/ML/Models/Finetuned/LiquidAI-LFM2.5-1.2B-Instruct/LiquidAI-LFM2.5-1.2B-Instruct-Code-Pro/sft_code_gguf\\LFM2.5-1.2B-Instruct.F16.gguf', 'D:/ML/Models/Finetuned/LiquidAI-LFM2.5-1.2B-Instruct/LiquidAI-LFM2.5-1.2B-Instruct-Code-Pro/sft_code_gguf\\LFM2.5-1.2B-Instruct.Q8_0.gguf', 'D:/ML/Models/Finetuned/LiquidAI-LFM2.5-1.2B-Instruct/LiquidAI-LFM2.5-1.2B-Instruct-Code-Pro/sft_code_gguf\\LFM2.5-1.2B

{'save_directory': 'D:/ML/Models/Finetuned/LiquidAI-LFM2.5-1.2B-Instruct/LiquidAI-LFM2.5-1.2B-Instruct-Code-Pro/sft_code',
 'gguf_directory': 'D:/ML/Models/Finetuned/LiquidAI-LFM2.5-1.2B-Instruct/LiquidAI-LFM2.5-1.2B-Instruct-Code-Pro/sft_code_gguf',
 'gguf_files': ['D:/ML/Models/Finetuned/LiquidAI-LFM2.5-1.2B-Instruct/LiquidAI-LFM2.5-1.2B-Instruct-Code-Pro/sft_code_gguf\\LFM2.5-1.2B-Instruct.F16.gguf',
  'D:/ML/Models/Finetuned/LiquidAI-LFM2.5-1.2B-Instruct/LiquidAI-LFM2.5-1.2B-Instruct-Code-Pro/sft_code_gguf\\LFM2.5-1.2B-Instruct.Q8_0.gguf',
  'D:/ML/Models/Finetuned/LiquidAI-LFM2.5-1.2B-Instruct/LiquidAI-LFM2.5-1.2B-Instruct-Code-Pro/sft_code_gguf\\LFM2.5-1.2B-Instruct.Q4_K_M.gguf'],
 'modelfile_location': None,
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}